# Pickleball Court-Coverage Heatmap

خط معالجة فيديو يحوّل لقطات المباراة من زاوية علوية إلى خريطة حرارية لتغطية الملعب.

**الخطوات:** رفع الفيديو ← قراءة خصائصه ← كشف اللاعبين بـ YOLO11x وبناء الخريطة الحرارية ← إعادة الترميز بـ H.264 ← العرض والتصدير.

شغّلي الخلايا بالترتيب من الأعلى، ويُفضَّل تشغيل GPU من Runtime ← Change runtime type.


In [1]:
# 🛠️ تثبيت مكتبة Ultralytics (تحتوي على YOLO وحلول solutions الجاهزة)
!pip install -q ultralytics
print("✓ تم التثبيت بنجاح")

# 📥 رفع الفيديو من جهازك مباشرة إلى بيئة Colab
from google.colab import files

uploaded = files.upload()  # 💡 يفتح نافذة لاختيار الملف من جهازك
video_path = list(uploaded.keys())[0]  # نأخذ اسم أول ملف تم رفعه
print(f"✓ تم رفع: {video_path}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.1/46.1 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 25.3 MB/s eta 0:00:00
✓ تم التثبيت بنجاح


Saving 13658128_2160_3840_24fps.mp4 to 13658128_2160_3840_24fps.mp4
✓ تم رفع: 13658128_2160_3840_24fps.mp4


In [2]:
# 🎬 استخراج خصائص الفيديو قبل ما نبدأ المعالجة
# لماذا؟ لأن المسجل (VideoWriter) لازم يعرف الأبعاد والسرعة لينتج فيديو سليم
import cv2

cap = cv2.VideoCapture(video_path)

# 1. استخراج الأبعاد (العرض والارتفاع بالبكسل)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# 2. استخراج السرعة (عدد الإطارات في الثانية)
fps = int(cap.get(cv2.CAP_PROP_FPS))

# 3. استخراج العدد الكلي للإطارات (لحساب نسبة التقدم لاحقاً)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

cap.release()  # دائماً نحرر مصدر الفيديو بعد الاستخدام

print(f"📐 الأبعاد: {w}×{h}")
print(f"⚡ السرعة: {fps} FPS")
print(f"🎞️ عدد الإطارات: {total_frames}")
print(f"⏱️ المدة: {total_frames/fps:.1f} ثانية")

📐 الأبعاد: 2160×3840
⚡ السرعة: 23 FPS
🎞️ عدد الإطارات: 715
⏱️ المدة: 31.1 ثانية


In [3]:
# 🔥 المرحلة الأساسية: بناء الخريطة الحرارية يدوياً
# 💡 ليش يدوياً؟ لأن solutions.Heatmap في الإصدار الحالي ما يقبل معامل imgsz
# (والتشخيص أكد إن imgsz=1920 ضروري لكشف اللاعبين بزاوية الدرون العمودية)
# الحل: نستخدم YOLO مباشرة (يقبل imgsz) ونبني الهيتماب بـ NumPy + OpenCV
import numpy as np
from ultralytics import YOLO

# 1. اسم ملف الفيديو الناتج (خام من OpenCV)
raw_video = "heatmap_raw.mp4"

# 2. تحسين الأداء: نعالج 1 من كل skip_rate إطارات
# 💡 الفيديو 23 FPS، نخفّضه لـ 7 FPS — اللاعبين ما يتحركون كثير في 0.13 ثانية
TARGET_FPS = 7
skip_rate = max(1, int(fps / TARGET_FPS))
print(f"⚡ تسريع: معالجة إطار كل {skip_rate} إطارات")

# 3. تحميل النموذج الأقوى (yolo11x ضروري لزاوية الدرون العمودية)
model = YOLO("yolo11x.pt")

# 4. إنشاء "مُجمّع الحرارة" — مصفوفة فارغة بنفس حجم الفيديو
# 💡 الفكرة الأساسية للهيتماب:
# - كل مرة YOLO يكتشف لاعب، نضيف بقعة بيضاء في موقعه على هذه المصفوفة
# - مع مرور الوقت، المناطق المتكررة تتراكم وتصير "ساخنة" (قيم عالية)
# - المناطق اللي ما داس فيها أحد تبقى صفر (باردة)
accumulator = np.zeros((h, w), dtype=np.float32)

# 5. فتح الفيديو وتجهيز المسجل
cap = cv2.VideoCapture(video_path)
writer = cv2.VideoWriter(raw_video, cv2.VideoWriter_fourcc(*'mp4v'), TARGET_FPS, (w, h))

# 6. حلقة المعالجة الرئيسية
frame_idx = 0
processed = 0
while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break

    if frame_idx % skip_rate == 0:
        # 🎯 الكشف بإعدادات التشخيص المؤكدة: imgsz=1920, conf=0.30
        results = model(frame, classes=[0], conf=0.30, imgsz=1920, verbose=False)

        # 📍 لكل لاعب مكتشف، نضيف بقعة بيضاء على المُجمّع
        for box in results[0].boxes.xyxy.cpu().numpy():
            x1, y1, x2, y2 = box.astype(int)
            cx = (x1 + x2) // 2                    # المركز الأفقي للاعب
            cy = (y1 + y2) // 2                    # المركز العمودي للاعب
            # نرسم دائرة قطرها 80 بكسل عند موقع اللاعب (قيمة 1.0 = حرارة كاملة)
            cv2.circle(accumulator, (cx, cy), 80, 1.0, -1)

        # 🌫️ تنعيم البقع بفلتر Gaussian — يخلي الهيتماب تطلع متدرجة بدل دوائر حادة
        smoothed = cv2.GaussianBlur(accumulator, (151, 151), 0)

        # 🌈 تحويل قيم الحرارة لألوان: تطبيع 0-255 ثم تطبيق سلم JET
        if smoothed.max() > 0:
            normalized = cv2.normalize(smoothed, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
        else:
            normalized = np.zeros_like(smoothed, dtype=np.uint8)
        colored = cv2.applyColorMap(normalized, cv2.COLORMAP_JET)

        # 💧  الهيتماب تظهر فقط في الأماكن الساخنة (مو على كل الصورة)
        # alpha = شدة الحرارة (0 = شفاف كامل، 0.6 = ظاهر 60%)
        alpha = (smoothed / smoothed.max() if smoothed.max() > 0 else smoothed)
        alpha = np.clip(alpha, 0, 1)[..., np.newaxis] * 0.6
        overlay = (frame.astype(np.float32) * (1 - alpha) +
                   colored.astype(np.float32) * alpha).astype(np.uint8)

        writer.write(overlay)
        processed += 1

        if processed % 20 == 0:
            progress = (frame_idx / total_frames) * 100
            print(f"معالجة: {processed} إطار محلّل ({progress:.0f}%)")

    frame_idx += 1

# 7. تحرير الموارد (مهم جداً وإلا الملف يطلع تالف)
cap.release()
writer.release()
print(f"✓ تمت معالجة {processed} إطار من أصل {frame_idx}")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
⚡ تسريع: معالجة إطار كل 3 إطارات
معالجة: 20 إطار محلّل (8%)
معالجة: 40 إطار محلّل (16%)
معالجة: 60 إطار محلّل (25%)
معالجة: 80 إطار محلّل (33%)
معالجة: 100 إطار محلّل (42%)
معالجة: 120 إطار محلّل (50%)
معالجة: 140 إطار محلّل (58%)
معالجة: 160 إطار محلّل (67%)
معالجة: 180 إطار محلّل (75%)
معالجة: 200 إطار محلّل (83%)
معالجة: 220 إطار محلّل (92%)
✓ تمت معالجة 239 إطار من أصل 715


In [8]:
# 🎥 لماذا هذه الخطوة؟
# OpenCV يحفظ الفيديو بـ codec mp4v، ومعظم متصفحات الويب (وColab) ما تدعمه.
# لذلك نستخدم FFmpeg لإعادة الترميز بـ H.264 (متوافق مع كل المتصفحات).
import os

final_video = "pickleball_heatmap.mp4"  # الملف النهائي المتوافق مع Colab
os.system(f"ffmpeg -i {raw_video} -vcodec libx264 -f mp4 {final_video} -y -loglevel quiet")
print("✓ تم التحويل — الفيديو جاهز للعرض في Colab")

✓ تم التحويل — الفيديو جاهز للعرض في Colab


In [9]:
!ffmpeg -i pickleball_heatmap.mp4 -t 8 \
  -vf "fps=8,scale=640:-1:flags=lanczos,split[s0][s1];[s0]palettegen[p];[s1][p]paletteuse" \
  heatmap.gif -y -loglevel quiet

import os
print("الحجم:", round(os.path.getsize("heatmap.gif")/1e6, 2), "MB")

from google.colab import files
files.download("heatmap.gif")

الحجم: 19.46 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# 📺 عرض الفيديو مباشرة داخل دفتر Colab
from IPython.display import Video

Video(final_video, embed=True, width=900)

In [11]:
from google.colab import files
files.download("pickleball_heatmap.mp4")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>